# Exercise2 solution:

In [ ]:
import netsquid as ns
ns.set_qstate_formalism(ns.QFormalism.DM)

## Network setup

In [ ]:
from netsquid.components import QuantumProcessor
from netsquid.components.models import FixedDelayModel, DepolarNoiseModel
from netsquid.components import PhysicalInstruction, INSTR_INIT, INSTR_H, INSTR_CNOT, INSTR_MEASURE

instr_init = PhysicalInstruction(INSTR_INIT,    duration=10)
instr_meas = PhysicalInstruction(INSTR_MEASURE, duration=10)
instr_h    = PhysicalInstruction(INSTR_H,       duration=10)
instr_cnot = PhysicalInstruction(INSTR_CNOT,    duration=10, topology=[(0, 1)])

qerror_model = DepolarNoiseModel(depolar_rate=1e7)

qprocessor = QuantumProcessor("Alice memory", 
                        num_positions=2,
                        memory_noise_models=qerror_model,
                        phys_instructions=[instr_init, instr_meas, instr_h, instr_cnot])

In [ ]:
from netsquid.nodes import Node
from netsquid.components import Port, QuantumChannel
from netsquid.nodes import DirectConnection
from netsquid.components.models import FixedDelayModel, DepolarNoiseModel, DephaseNoiseModel

alice_node = Node("Alice", qmemory=qprocessor)
bob_node   = Node("Bob")

alice_port = Port("Alice_port", alice_node)
bob_port   = Port("Bob_port",   bob_node)

delay_model = FixedDelayModel(delay=10.)
error_model = DephaseNoiseModel(dephase_rate=0.2, time_independent=True)

channel_a_to_b = QuantumChannel("Channel Alice to Bob",
                                  models={"delay_model": delay_model,
                                          "quantum_noise_model": error_model}
                                  )

channel_b_to_a = QuantumChannel("Channel Bob to Alice",
                                  models={"delay_model": delay_model,
                                          "quantum_noise_model": error_model}
                                  )


connection = DirectConnection(name="connection",
                              channel_AtoB=channel_a_to_b,
                              channel_BtoA=channel_b_to_a)


alice_node.connect_to(remote_node=bob_node, connection=connection,
                     local_port_name="Alice_port", remote_port_name="Bob_port")

## Protocols

In [ ]:
from netsquid.components import QuantumProgram

class BellStateCreateProgram(QuantumProgram):
    default_num_qubits = 2
    
    def program(self):
        q1, q2 = self.get_qubit_indices(2)
        self.apply(INSTR_INIT, q1)
        self.apply(INSTR_INIT, q2)
        self.apply(INSTR_H   , q1)
        self.apply(INSTR_CNOT, [q1, q2])
        yield self.run()

bellstate_create_program = BellStateCreateProgram()

In [ ]:
from netsquid.protocols import NodeProtocol

class AliceProtocol(NodeProtocol):
    def run(self):
        port = self.node.ports["Alice_port"]

        yield self.node.qmemory.execute_program(bellstate_create_program, qubit_mapping=[0,1])

        q2, = self.node.qmemory.pop(positions=1)
                
        port.tx_output(q2)
        print(f"{ns.sim_time()} ns Alice send q2")

        # Wait 30 ns
        yield self.await_timer(30)

        q1, = self.node.qmemory.peek(positions=0)
        dm = ns.qubits.reduced_dm(q1.qstate.qubits)
        print(f"{ns.sim_time()} ns Alice stops waiting, DM=\n{dm}")
        
        m, _ = self.node.qmemory.measure(positions=0, discard=True)

        print(f"{ns.sim_time()} ns Alice measures q1: {m}")



class BobProtocol(NodeProtocol):
    def run(self):
        port = self.node.ports["Bob_port"]

        yield self.await_port_input(port)
        
        q2 = port.rx_input().items[0]

        dm = ns.qubits.reduced_dm(q2.qstate.qubits)
        print(f"{ns.sim_time()} ns Bob receives q2, DM=\n{dm}")

        m, _ = ns.qubits.measure(q2, keep_combined=True)

        print(f"{ns.sim_time()} ns Bob measures q2: {m}")

alice_protocol = AliceProtocol(node=alice_node)
bob_protocol = BobProtocol(node=bob_node)

## Run simulation

In [ ]:
# Start all protocols
alice_protocol.start()
bob_protocol.start()

sim_stats = ns.sim_run()

print(sim_stats)

# Reset simulation
alice_protocol.stop()
bob_protocol.stop()

ns.sim_reset()